In [ ]:
# ============================================================================
# Cell 1: 导入库
# ============================================================================
from vnpy.trader.constant import Interval
from vnpy.alpha.strategy import BacktestingEngine2
from vnpy.alpha.lab import AlphaLab
import vnpy.alpha.strategy.strategies.equity_demo_strategy2 as equity_demo_strategy2
import importlib
from datetime import datetime,timedelta
from pathlib import Path
# 重载策略类
importlib.reload(equity_demo_strategy2)
EquityDemoStrategy = equity_demo_strategy2.EquityDemoStrategy2  # 重载策略类

In [ ]:
# ============================================================================
# Cell 2: 路径配置和AlphaLab创建
# ============================================================================
vt_index_symbol = "000300.SSE"
BASE_PATH = Path('D:/Aquant project/MF')
LAB_PATH = BASE_PATH / 'MF_lab'

# 获取MF_Lab
lab = AlphaLab(str(LAB_PATH))

In [ ]:
# ============================================================================
# Cell 3: 时间配置
# ============================================================================
# 总时间跨度
start = datetime(2018, 1, 1)
end = end = datetime(2026,5,8)
interval1 = Interval.MINUTE                  #数据频率

# 回测跨度 回测需要日线数据算收益
test_start = datetime(2025, 1, 1)
test_end = end
interval2 = Interval.DAILY

# 训练跨度
train_start = datetime(2018, 1, 1)
train_end = datetime(2023, 12, 31)

# 验证跨度
valid_start = datetime(2024, 1, 1)
valid_end = datetime(2024, 12, 31)

# 加载成分股代码
component_symbols = lab.load_component_symbols(vt_index_symbol, test_start, test_end)
# for vt_symbol in component_symbols:
#     lab.add_contract_setting(
#         vt_symbol,
#         long_rate=5/10000,
#         short_rate=15/10000,
#         size=1,
#         pricetick=0.0001
#     )

In [ ]:
# ============================================================================
# Cell 4: 加载信号
# ============================================================================
signal = lab.load_signal("v100")
import polars as pl

In [ ]:
print(signal)

In [ ]:
# ============================================================================
# Cell 5: 创建回测引擎对象
# ============================================================================
slippage = 0.0006
engine = BacktestingEngine2(lab)
# 设置回测参数
engine.set_parameters(
    vt_symbols=component_symbols,
    interval=Interval.DAILY,
    start=test_start,
    end=test_end,
    capital=100000,
    min_commission = 5,
    slippage = slippage,
    adjust_type = 'none'
)

# 添加策略实例
setting = {"top_k":10, "min_days":6, "cash_ratio": 1, "open_rate": 0.0005, "close_rate": 0.0015, "slippage": slippage}
engine.add_strategy(EquityDemoStrategy, setting, signal)

In [ ]:
# 执行回测任务
engine.load_data()
engine.run_backtesting()
engine.calculate_result()
engine.calculate_statistics()
engine.show_chart()


In [ ]:
# 显示超额收益分析结果
engine.show_performance(benchmark_symbol=vt_index_symbol)

In [ ]:
# print(engine.trades)

In [ ]:
# print(engine.logs)

In [ ]:
# with pl.Config(tbl_rows=317, tbl_cols=None, fmt_str_lengths=None):
#     display(engine.daily_df)